In [ ]:
# pip install transformers torch pillow requests

import torch
import requests
from PIL import Image
from transformers import AutoProcessor, AutoModel
from torch.nn.functional import cosine_similarity

# Load I-JEPA model
model_id = "facebook/ijepa_vith14_1k"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id).eval()

# Helper function to get embeddings
@torch.no_grad()
def get_embedding(image_url):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"}
    response = requests.get(image_url, headers=headers, stream=True)
    response.raise_for_status()
    image = Image.open(response.raw)
    inputs = processor(image, return_tensors="pt")
    outputs = model(**inputs)
    # Mean pool over patch tokens to get a single vector
    return outputs.last_hidden_state.mean(dim=1)

# Standard and robust image URLs
url_dog = "https://images.unsplash.com/photo-1543466835-00a7907e9de1?q=80&w=1000&auto=format&fit=crop"
url_cat = "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?q=80&w=1000&auto=format&fit=crop"

embed_dog = get_embedding(url_dog)
embed_cat = get_embedding(url_cat)

# Compute cosine similarity
sim = cosine_similarity(embed_dog, embed_cat)
print(f"Dog vs Cat similarity: {sim.item():.4f}")

# Alternate reliable dog image URL
url_dog2 = "https://images.unsplash.com/photo-1583511655857-d19b40a7a54e?q=80&w=1000&auto=format&fit=crop"
embed_dog2 = get_embedding(url_dog2)
sim2 = cosine_similarity(embed_dog, embed_dog2)
print(f"Dog vs Dog similarity: {sim2.item():.4f}")

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Dog vs Cat similarity: 0.3244
Dog vs Dog similarity: 0.3804


In [ ]:
# Running similarity checks to display the outputs
embed_dog = get_embedding(url_dog)
embed_cat = get_embedding(url_cat)

# Compute cosine similarity
sim = cosine_similarity(embed_dog, embed_cat)
print(f"Dog vs Cat similarity: {sim.item():.4f}")

# Compare dog to another dog
url_dog2 = "https://images.unsplash.com/photo-1583511655857-d19b40a7a54e?q=80&w=1000&auto=format&fit=crop"
embed_dog2 = get_embedding(url_dog2)
sim2 = cosine_similarity(embed_dog, embed_dog2)
print(f"Dog vs Dog similarity: {sim2.item():.4f}")

Dog vs Cat similarity: 0.3244
Dog vs Dog similarity: 0.3804
